# 실습 6: 도로 균열 유무를 분류하는 합성곱 신경망 모델 구현하기

**문제 상황** 폭염과 폭우로 전국 각지에서 아스팔트가 깔린 도로에 금이 가고 함몰되거나 녹아 내리는 등의 도로 균열 현상이 나타나고 있다. 이러한 도로 균열은 자동차 운전자가 주행 중에 발견하기 어렵고 사고 발생 후 인식하게 되므로 교통 정체, 차량 손상, 급제동뿐만 아니라 뒤따르던 차량의 추돌 등 큰 사고로도 이어질 수 있다.

| 단계 | 과정 | 처리 내용 |
|:---:|---|---|
| 단계 1 | 문제 정의하기 | 도로 균열 유무를 판단해 보자. |
| 단계 2 | 데이터 수집 및 전처리하기 | (1) 데이터 수집하기<br>(2) 데이터 전처리하기 `ImageDataGenerator()` |
| 단계 3 | 신경망 모델 생성하기 | (1) 이미지 특징 추출하기(VGG-16) `vgg=VGG16()`<br>(2) 완전 연결 계층 설계하기 `model=Sequential()`<br>(3) 신경망 모델 환경 설정하기 `model.compile()`<br>(4) 신경망 모델 학습하기 `model.fit()` |
| 단계 4 | 신경망 모델 평가 및 예측하기 | (1) 신경망 모델 평가하기 `model.evaluate()`<br>(2) 신경망 모델 예측하기 `model.predict()` |

## 단계 0: 준비 (라이브러리 설치)

In [ ]:
import importlib, sys, subprocess

packages = [
    ('tensorflow', 'tensorflow'),
    ('numpy', 'numpy'),
    ('matplotlib.pyplot', 'matplotlib'),
    ('seaborn', 'seaborn'),
    ('sklearn', 'scikit-learn'),
]

for module_name, pip_name in packages:
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

from sklearn import set_config
set_config(display="text")

print('라이브러리 준비 완료')

## 단계 1 문제 정의하기

사전 훈련된 합성곱 신경망 모델 중 VGG16 모델로 도로의 균열 유무를 판단해 보자.

VGG16: 2014년에 발표된 합성곱 신경망 모델로, 16개의 합성곱 층으로 구성됨.

## 단계 2 데이터 수집 및 전처리하기

### (1) 데이터 수집하기

캐글 사이트의 도로 균열 이미지 데이터셋(https://www.kaggle.com/datasets/arunrk7/surface-crack-detection)은 균열된 이미지(Positive)와 균열되지 않은 이미지(Negative) 폴더로 저장되어 있다. 캐글 사이트에 데이터가 너무 많으므로 주어진 링크로 들어가서 crack.zip 파일을 다운로드한다.

crack.zip은 훈련(train), 테스트(test) 폴더를 만들어 이미지 파일을 일부 이동시킨 것이다.

https://bit.ly/crack_dataset

- 훈련(train): 300개
- 테스트(test): 80개
- 레이블: Negative, Positive

### (2) 데이터 전처리하기

실행 환경에 따라 이미지 폴더에 접근한다.

- Google Colab: 내 드라이브 ＞ data ＞ crack
- 로컬 환경: 현재 폴더 ＞ crack

각 `crack` 폴더 안에는 `train`, `test` 폴더가 있어야 한다.

① 실행 환경에 맞는 데이터 폴더로 이동한다.

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    current_dir = Path.cwd()
    data_dir = current_dir if current_dir.name == 'crack' else current_dir / 'crack'
else:
    drive.mount('/content/gdrive')
    data_dir = Path('/content/gdrive/My Drive/data/crack')

os.chdir(data_dir)
print('현재 폴더:', Path.cwd())

② ImageDataGenerator를 이용하여 훈련, 테스트 이미지 데이터를 전처리할 수 있다.

In [ ]:
# ImageDataGenerator를 생성하기(이미지 픽셀값을 0~1 사이 범위로 조정)
# train 폴더의 이미지 데이터를 불러와 전처리 적용하기
# 이미지 훈련 데이터 폴더명(train)


**❓ 확인하기**

- 훈련 폴더에서 불러온 이미지는 몇 개인가?  → ( )개
- 분류할 클래스는 몇 개인가?  → ( )개
- 훈련 데이터가 준비되었으므로, 모델의 성능을 나중에 확인하려면 다음으로 어떤 데이터가 필요한가?  → ( )

훈련 데이터(train)는 총 300개이다.

In [ ]:
# ImageDataGenerator를 생성하기(이미지 픽셀값을 0~1 사이 범위로 조정)
# 주어진 폴더(test)에서 이미지 데이터를 불러와 전처리 적용하기


**❓ 확인하기**

- 테스트 폴더에서 불러온 이미지는 몇 개인가?  → ( )개
- 테스트 이미지의 순서를 섞지 않은 이유는 무엇인가?

  → ( )
- 훈련·테스트 이미지가 준비되었으므로, 다음 단계에서 이미지로부터 무엇을 추출해야 하는가?  → ( )

테스트 데이터는 총 80개이다.

## 단계 3 신경망 모델 생성하기

합성곱 신경망 모델로 이미지를 분류하기 위해서는 이미지의 특징을 추출하고, 추출된 특징값을 완전 연결 계층의 입력층으로 받아 분류 모델을 수행한다.

### (1) 이미지 특징 추출하기

VGG16 모델을 이용해 이미지 특징을 추출한다.

사전 훈련된 합성곱 신경망 모델인 VGG16을 불러와 이미지의 특징을 추출한다. include_top=False를 설정해, VGG16 모델의 완전 연결 계층을 제외한 합성곱 층, 풀링 층만 실행해 특징을 추출한다.

In [ ]:
# layer.trainable = False: VGG16 모델의 사전 학습된 가중치를 변경하지 않고 그대로 사용


**❓ 확인하기**

- VGG16의 마지막 출력 형태는 무엇인가?  → `( )`
- `Trainable params`가 0으로 표시되는 이유는 무엇인가?

  → ( )
- 추출된 특징으로 도로 균열을 분류하려면 다음으로 어떤 계층을 연결해야 하는가?  → ( )

### (2) 완전 연결 계층 설계하기

추출된 특징값을 입력층으로, 완전 연결 계층으로 신경망 모델을 설계한다.

① 케라스 라이브러리로 완전 연결 계층을 만들기 위한 클래스를 불러온다.

In [ ]:
# keras.models의 Sequential 불러오기
# keras.layers의 Dense, Flatten 불러오기


② VGG16으로 특징을 추출한 결과(vgg)를 완전 연결 계층에 연결한다. 두 가지로 분류하기 위해 다음과 같이 설계한다.

In [ ]:
# 은닉층 추가하기(64개 노드, 활성화 함수 relu)
# 출력층 추가하기(2개 노드, 활성화 함수 softmax)


**❓ 확인하기**

- 모델의 최종 출력 형태에서 마지막 숫자가 2인 이유는 무엇인가?

  → ( )
- 전체 파라미터 중 실제로 학습되는 파라미터는 어느 부분의 파라미터인가?

  → ( )
- 모델 구조를 완성했으므로 학습 전에 무엇을 설정해야 하는가?  → ( )

### (3) 신경망 모델 환경 설정하기

분류 모델에 대한 손실함수(loss)와 최적화 함수(optimizer)를 설정한다.

이미지 분류 모델에서 손실함수는 ‘categorical_crossentropy’, 최적화 함수는 ‘adam’, 평가 지표는 정확도(accuracy)로 설정하였다.

### (4) 신경망 모델 학습하기

훈련 데이터(training_set)로 신경망 모델을 학습한다.

**❓ 확인하기**

- 5번의 반복 학습 동안 손실(loss)과 정확도(accuracy)는 어떻게 변하는가?

  → ( )
- 훈련 정확도가 높으면 테스트 이미지도 반드시 같은 정확도로 분류할까?

  → ( )

훈련 데이터로 모델을 훈련한 결과, 분류 정확도가 높음을 알 수 있다.

## 단계 4 신경망 모델 평가 및 예측하기

### (1) 모델 성능 평가하기

테스트 데이터(test_set)로 모델의 성능을 평가한다.

**❓ 확인하기**

- 테스트 데이터의 손실(loss)과 정확도(accuracy)는 각각 얼마인가?  → ( )
- 훈련 정확도와 테스트 정확도의 차이가 크다면 무엇을 의심할 수 있는가?  → ( )
- 전체 정확도만으로는 어떤 이미지를 잘못 분류했는지 알 수 있는가?

  → ( )

테스트 데이터로 모델 성능을 평가한 결과, 분류 정확도가 높음을 알 수 있다.

### (2) 예측하기

① 테스트 데이터(test_set)로 예측을 수행한다.

② 데이터의 클래스값을 확인한다. 균열이 없는 ‘Negative’는 0, 균열이 있는 ‘Positive’는 1이다.

In [ ]:
# 훈련 데이터셋의 클래스 레이블과 인덱스 확인하기


**❓ 확인하기**

- 균열이 없는 `Negative`와 균열이 있는 `Positive`의 클래스값은 각각 얼마인가?  → Negative ( ) · Positive ( )
- 예측값과 실제값을 클래스별로 비교해 어떤 오류가 발생했는지 확인하려면 무엇을 만들면 되는가?  → ( )

③ 예측 결과와 테스트 데이터의 레이블값으로 혼동 행렬을 구한다.

In [ ]:
# 혼동 행렬 함수 불러오기
# 예측 결과에서 가장 큰 인덱스값 추출하기
# 혼동 행렬 계산하기(실젯값, 예측값)
# 혼동 행렬 표시하기
# 제목, x축 이름(Predicted), y축 이름(Actual) 설정하기


**❓ 확인하기**

- 혼동 행렬의 대각선 값은 무엇을 의미하는가?  → ( )
- 교과서 실행 예에서 정확히 분류한 이미지는 모두 몇 개인가?  → 40 + 33 = ( )개
- 가장 많이 발생한 오류는 무엇인가?

  → ( )

## 🏁 마무리: 스스로 정리하기

1. 훈련 데이터와 테스트 데이터는 각각 몇 개이며, 분류할 클래스는 무엇인가?

   →

2. VGG16의 모든 층을 다시 학습하지 않고 가중치를 고정한 이유는 무엇인가?

   →

3. 테스트 정확도와 혼동 행렬을 함께 확인해야 하는 이유는 무엇인가?

   →

4. 교과서 실행 예에서 모델이 더 보완해야 할 분류는 무엇인가?

   →
